In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE itransition.default.parquet_df_with_price AS
    SELECT
        *,
        CASE
            -- ¢ belgisi bor
            WHEN unit_price RLIKE '.*¢.*' THEN
                CAST(COALESCE(NULLIF(regexp_replace(unit_price, '[^0-9]', ''), ''), '0') AS DECIMAL(10,2)) / 100
                * IF(upper(unit_price) RLIKE '.*(€|EUR).*', 1.2, 1.0)
            ELSE
                CAST(COALESCE(NULLIF(regexp_replace(unit_price, '[^0-9.]', ''), ''), '0') AS DECIMAL(10,2))
                * IF(upper(unit_price) RLIKE '.*(€|EUR).*', 1.2, 1.0)
        END AS paid_price
    FROM itransition.default.parquet_df
    WHERE unit_price IS NOT NULL
""")

print("Jadval saqlandi!")

# Tekshirish
display(
    spark.table("itransition.default.parquet_df_with_price")
    .select("unit_price", "paid_price")
    .limit(10)
)

In [0]:
from dateutil import parser as dateparser
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# UDF yaratish
@F.udf(returnType=DateType())
def parse_date(ts):
    if ts is None:
        return None
    try:
        return dateparser.parse(ts).date()
    except:
        try:
            cleaned = ts.replace(",", " ")
            return dateparser.parse(cleaned, dayfirst=True).date()
        except:
            try:
                cleaned = ts.replace(",", " ")
                return dateparser.parse(cleaned).date()
            except:
                return None

# Jadvalga qo'llash
df = spark.table("itransition.default.parquet_df")

df_fixed = (
    df
    .withColumn("date",  parse_date(F.col("timestamp")))
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day",   F.dayofmonth("date"))
)

# Avval tekshirish
null_count = df_fixed.filter(F.col("date").isNull()).count()
print(f"NULL qolganlar: {null_count}")

# To'g'ri bo'lsa jadvalga saqlash
df_fixed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("itransition.default.parquet_df_with_price")

print("Jadval yangilandi!")

In [0]:
from dateutil import parser as dateparser
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# UDF yaratish
@F.udf(returnType=DateType())
def parse_date(ts):
    if ts is None:
        return None
    try:
        return dateparser.parse(ts).date()
    except:
        try:
            cleaned = ts.replace(",", " ")
            return dateparser.parse(cleaned, dayfirst=True).date()
        except:
            try:
                cleaned = ts.replace(",", " ")
                return dateparser.parse(cleaned).date()
            except:
                return None

# 1. Asl jadvaldan o'qish
df = spark.table("itransition.default.parquet_df")

# 2. date, year, month, day qo'shish
df_fixed = (
    df
    .withColumn("date",  parse_date(F.col("timestamp")))
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day",   F.dayofmonth("date"))
)

# 3. paid_price qo'shish
df_fixed.createOrReplaceTempView("temp_df")

df_final = spark.sql("""
    SELECT
        *,
        CASE
            WHEN unit_price IS NULL THEN NULL
            WHEN unit_price RLIKE '.*¢.*' THEN
                CAST(COALESCE(NULLIF(regexp_replace(unit_price, '[^0-9]', ''), ''), '0') AS DECIMAL(10,2)) / 100
                * IF(upper(unit_price) RLIKE '.*(€|EUR).*', 1.2, 1.0)
            ELSE
                CAST(COALESCE(NULLIF(regexp_replace(unit_price, '[^0-9.]', ''), ''), '0') AS DECIMAL(10,2))
                * IF(upper(unit_price) RLIKE '.*(€|EUR).*', 1.2, 1.0)
        END AS paid_price
    FROM temp_df
""")

# 4. Tekshirish
null_count = df_final.filter(F.col("date").isNull()).count()
print(f"NULL qolganlar: {null_count}")

# 5. Jadvalga saqlash
df_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("itransition.default.parquet_df_with_price")

print("Jadval saqlandi!")

# 6. Natijani ko'rish
display(
    spark.table("itransition.default.parquet_df_with_price")
    .select("timestamp", "date", "year", "month", "day", "unit_price", "paid_price")
    .limit(10)
)

In [0]:
spark.sql("select * from itransition.default.parquet_df_with_price limit 20").display()